# 02. Mineração de Dados e Avaliação

Este notebook faz parte da etapa de **Mineração de Dados e Avaliação** do nosso projeto. O objetivo principal é encontrar gastos fora do padrão (anomalias) usando o Cartão de Pagamento do Governo Federal (CPGF). Como não sabemos de antemão quais gastos são fraudes, vamos usar **Detecção de Outliers** com técnicas de agrupamento (modelos não supervisionados).

## 1. Introdução e Carga de Dados

Primeiro, vamos carregar os dados que já limpamos passados (`cpgf_normalizado.csv`). As colunas com valores comportamentais (como o total gasto, média, etc.) já foram padronizadas usando o Z-Score (StandardScaler). Isso significa que elas estão na mesma escala, o que ajuda muito os algoritmos que calculam distância entre os pontos, como o K-Means e o HDBSCAN.

A nossa missão aqui é simples: encontrar grupos de CPFs com gastos comuns e isolar aqueles que têm um comportamento muito diferente da regra. O índice original da nossa tabela (`CPF PORTADOR`) precisa ser mantido para sabermos investigar essas anomalias depois no pós-processamento.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import hdbscan
import os

# Configurando o estilo visual das plotagens
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

# Caminho dos dados importados do pré-processamento
data_path = '../dados/cpgf_normalizado.csv'

# Carga dos dados garantindo a restauração do índice original (CPF PORTADOR)
df = pd.read_csv(data_path, index_col=0)

# Verificando as dimensões e os dados importados
display(df.head().style.set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}, {'selector': 'td', 'props': [('text-align', 'center')]}]).set_properties(**{'background-color': '#f8f9fa', 'color': 'black', 'border': '1px solid black'}))
print(f"📌 Dataset pronto para mineração! Dimensões totais: {df.shape[0]} CPFs (linhas) e {df.shape[1]} features (colunas).")


""
CPF PORTADOR;total_gasto;media_gasto;qtd_transacoes;max_gasto
***.000.021-**;0.08867853263573086;-0.16079653225920068;1.2954522078595712;-0.24549051454515236
***.000.060-**;-0.05467801108072213;0.06281762951271412;-0.28635045726554104;-0.23950814957595423
***.000.230-**;-0.052217409394931275;-0.21298814284797868;-0.19647530583797784;-0.2681166734822267
***.000.251-**;-0.053432898711539265;-0.019894025230132686;-0.27736294212278473;-0.20275933619373712
***.000.379-**;-0.04577649031720023;-0.1321288883978725;-0.19647530583797784;-0.16002815784232188


Dimensões do dataset de análise: (9922, 0)


## 2. O Modelo Básico: K-Means

O **K-Means** é um dos algoritmos de agrupamento mais famosos e fáceis de usar. Ele divide os dados em um número fixo de grupos (chamado de $K$). Ele cria "centros" (centróides) para cada grupo e coloca cada linha no grupo do centro mais próximo a ela.

Apesar de ser muito rápido, ele tem um problema: ele assume que todos os grupos são redondos e têm tamanhos parecidos. Isso acaba forçando a separação de grupos que não deveriam ser separados, apenas para ficarem do mesmo tamanho esfericamente.

**Como vamos achar as anomalias com ele?** Depois que o K-Means separar os grupos, nós vamos medir qual é a distância de cada CPF até o centro do seu grupo (a distância Euclidiana). Se a pessoa ficou muito longe do centro, então os gastos dela foram muito diferentes do restante. Essas são nossas suspeitas principais neste modelo básico.

### 2.1. Escolhendo o número de grupos (Método do Cotovelo)

Para usar o K-Means, nós somos obrigados a dizer para ele quantos grupos ($K$) ele deve criar. Nós usamos o Método do Cotovelo (*Elbow Method*) pra descobrir isso de forma mais inteligente. Nós rodamos o K-Means várias vezes com diferentes números de grupos, e escolhemos o ponto no gráfico em que a curva parece formar um "cotovelo". Isso indica que não vale a pena adicionar mais grupos a partir dali.

In [4]:
# Definindo o intervalo para K de acordo com as diretrizes do notebook
k_range = range(2, 11)
sse = []

# Computando a soma dos erros quadráticos (inertia_)
for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans_temp.fit(df)
    sse.append(kmeans_temp.inertia_)

# Plotando graficamente o Método do Cotovelo
plt.figure()
plt.plot(k_range, sse, marker='o', linestyle='-', linewidth=2, markersize=8)
plt.title('Método do Cotovelo para Escolha de K Ótimo', fontsize=14)
plt.xlabel('Número de Grupos (K)')
plt.ylabel('Soma dos Erros (SSE)')
plt.xticks(k_range)
plt.show()

ValueError: at least one array or dtype is required

### 2.2. Treinando o Modelo K-Means

Neste nosso teste de baseline, vamos fixar nosso modelo em $K=4$, como se estivéssemos dividindo as despesas do governo em 4 perfis grandes (por exemplo: Muito baixo, Médio constante, Alto e Altíssimo).

Assim que o agrupamento tiver fim, nós calculamos e guardamos numa nova coluna exatamente essa distância para verificar o nível de estranheza de cada gasto baseando-se no centro.

In [ ]:
# Parâmetro global com K=4
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')

# Criamos a coluna dos clusters do modelo
df['Cluster_KMeans'] = kmeans.fit_predict(df)

# Agora, vamos medir quão distante cada um ficou
centroides = kmeans.cluster_centers_
distancias = []

for i in range(len(df)):
    cluster_idx = df['Cluster_KMeans'].iloc[i]
    ponto = df.drop(columns=['Cluster_KMeans']).iloc[i].values
    centroide = centroides[cluster_idx]
    
    # Distância Euclidiana
    dist = np.linalg.norm(ponto - centroide)
    distancias.append(dist)

df['Distancia_Centroide_KMeans'] = distancias

# Plotando para ver as pessoas muito longe do centro
plt.figure()
sns.histplot(df['Distancia_Centroide_KMeans'], bins=50, kde=True, color='crimson')
plt.title('Distribuição da Distância em relação aos Grupos', fontsize=14)
plt.xlabel('Distância Euclidiana (Quanto maior, mais anormal)')
plt.ylabel('Quantidade de CPFs')
plt.show()

print("Top 5 maiores anomalias segundo K-Means (Muito longe da norma):")
display(df.sort_values('Distancia_Centroide_KMeans', ascending=False).head())

## 3. O Modelo Avançado: HDBSCAN

O K-Means tem uma grande limitação: ele sofre com formatos irregulares de dados. Na vida real e no setor público, os dados não costumam se comportar em esferas perfeitas. Uma alternativa clássica é o algoritmo DBSCAN, que foca na densidade ao invés da distância, sendo excelente em achar formatos estranhos.

O grande problema do DBSCAN antigo é que num mesmo grupo governamental você pode ter muita diferença de concentração de gastos. Por exemplo: um Ministério inteiro concentrando muitos gastos num padrão comum, e outros órgãos com poucos e espalhados pagamentos. O DBSCAN exige uma configuração de raio fixa que unifica tudo de forma bruta e acaba errando nessas matrizes com densidade desigual.

A maior recomendação para isso hoje em dia é o incrível **HDBSCAN**. Ele arruma justamente o maior problema da densidade permitindo uma estrutura flexível (ele entende simultaneamente grupos fáceis muito densos e também menos densos).

Nessa nossa tarefa para a Detecção de Fraude, optaremos por ele justamente por nos dar duas vantagens absurdas:
1. Usuários de Cartões cujos gastos são tão aleatórios e isolados que não interagem com perfil comum nenhum não serão forçados a estarem dentro de um cluster. Eles ficam em **ruído**, rotulados de forma especial como `-1`.
2. Ele cria uma pontuação contínua interna de anomalias chamada `outlier_scores_` calculada baseada na proximidade. Quanto mais alta a probabilidade pontuada, maior o alerta de fraude.

In [ ]:
# Criando o HDBSCAN
# min_cluster_size=15 manda o agrupador ignorar grupos menores de 15 como algo válido
# min_samples=5 controla o tamanho da vizinhança na densidade
modelo_hdbscan = hdbscan.HDBSCAN(
    min_cluster_size=15,
    min_samples=5,
    gen_min_span_tree=True
)

# Usaremos as features limpas de volta, sem a coluna recém criada das distâncias do k-means
features_X = df.drop(columns=['Cluster_KMeans', 'Distancia_Centroide_KMeans'])

# Treinando e agrupando
modelo_hdbscan.fit(features_X)

# Salvando as previsões do modelo no nosso dataframe
df['Cluster_HDBSCAN'] = modelo_hdbscan.labels_
df['HDBSCAN_Outlier_Score'] = modelo_hdbscan.outlier_scores_

# Olhando os resultados das análises de rótulo
ruido_abs = len(df[df['Cluster_HDBSCAN'] == -1])
clusters_ativos = df['Cluster_HDBSCAN'].nunique() - 1

print(f"Número de Grupos válidos encontrados pelo HDBSCAN: {clusters_ativos}")
print(f"Número de gastos caóticos isolados na zona de Ruído (-1): {ruido_abs} CPFs ({ruido_abs/len(df):.2%}).\n")

# Vamos olhar pro gráfico das pontuações matemáticas atípicas de cada CPF
plt.figure()
sns.histplot(df['HDBSCAN_Outlier_Score'], bins=50, kde=True, color='indigo')
plt.title('Probabilidade de ser Anomalia pelo HDBSCAN', fontsize=14)
plt.xlabel('Pontuação HDBSCAN_Outlier_Score')
plt.ylabel('Frequência de CPFs')
plt.show()

print("Top 5 maiores anomalias baseadas no Isolamento Geográfico (HDBSCAN):")
display(df.sort_values(by='HDBSCAN_Outlier_Score', ascending=False).head())

## 4. Avaliação Técnica

Na detecção de anomalias sem rótulos (Não Supervisionada), nós não temos como saber a resposta certa validando a Fraude. Por isso, a gente vai medir a qualidade dos nossos agrupamentos pela distribuição lógica matemática usando o **Coeficiente de Silhueta (Silhouette Score)**.

Essa métrica avalia os agrupamentos com uma nota de $-1$ até a perfeição absoluta em $+1$, verificando:
1. **Coesão Interna**: Um membro de grupo deve ter gastos parecidos e estar bem grudado junto à média desse grupo dele.
2. **Separação**: Tem que estar separadamente bem mais distante dos perfis dos outros grupos que são diferentes dele.

Vale uma ressalva importante com o HDBSCAN: nós precisamos **filtrar o Ruído (`-1`)** impiedosamente fora do cálculo da Silhueta. Pela própria lógica, o erro isolado não tem perfil nenhum nem pertence a lugar nenhum (por isso não é um agrupamento natural); adicioná-los para a métrica despencaria a nossa nota atoa sem representar a verdade do modelo.

In [ ]:
# Pegamos só os dados originais
X = df.drop(columns=['Cluster_KMeans', 'Distancia_Centroide_KMeans', 'Cluster_HDBSCAN', 'HDBSCAN_Outlier_Score']).values

# Calculando a Silhueta básica para todos os dados agrupados pelo KMeans
silhueta_global_kmeans = silhouette_score(X, df['Cluster_KMeans'])
print(f"Nota de Silhueta do Modelo K-Means: {silhueta_global_kmeans:.4f}")

# Retirando os Ruídos (rótulo -1) salvando apenas CPFs de grupos coerentes do HDBSCAN pra avaliar
mascara_filtragem_hdbscan = (df['Cluster_HDBSCAN'] != -1)

X_purificado = X[mascara_filtragem_hdbscan]
labels_purificados = df.loc[mascara_filtragem_hdbscan, 'Cluster_HDBSCAN']

if labels_purificados.nunique() > 1:
    silhueta_limpa_hdbscan = silhouette_score(X_purificado, labels_purificados)
    print(f"Nota de Silhueta limpa sem Ruídos do Modelo HDBSCAN: {silhueta_limpa_hdbscan:.4f}")
else:
    print("Atenção: HDBSCAN gerou menos de 2 grupos estruturais e a Silhueta não pôde ser calculada.")

## 5. Exportação dos Resultados

Ufa! Terminamos a melhor parte. Chegamos no final colando toda nossa Base Original de atributos (`CPF PORTADOR` intacto no índice) unificada com os rótulos de perfil comportamental e pontuações de perigo descobertos e criados aqui dentro pelas análises da Mineração.

Gravaremos tudo agora no arquivo final `cpgf_minerado.csv` para podermos prosseguir diretamente para explorar as anomalias no nosso código de **Pós-processamento de Insights**.

In [ ]:
# Exportando nosso agrupamento com os resultados novos direto na pasta de dados do projeto
output_caminho = '../dados/cpgf_minerado.csv'

df.to_csv(output_caminho)

print(f"Dados com previsões de anomalias salvos com sucesso no endereço '{output_caminho}'.")
display(df.info())